# 01 -- The Raw Frames: inventory and health check

Every science image in the pipeline is a **multi-extension FITS** file:

| Plane | HDU | Meaning |
|-------|-----|---------|
| `SCI` | primary | science data (electrons; Jy after flux calibration) |
| `ERR` | ext `ERR` | 1-sigma per-pixel uncertainty |
| `DQ`  | ext `DQ` | integer quality bitmask |

Phase 1 (notebook 02) is what turns raw frames into that three-plane file, and
you will see the mechanics on a real calibrated frame there. This notebook is
about what goes *in*: an inventory of the raw frames on disk, and then a health
check that says whether they are worth reducing at all.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG
# ============================================================
# One shared module rather than this cell copied into six notebooks, so a
# path is changed once and the notebooks cannot drift apart.
# Override any path with an environment variable; see workshop_config.py.
import importlib, os, sys

_here = os.path.dirname(os.path.abspath('workshop_config.py'))
if _here not in sys.path:
    sys.path.insert(0, _here)

# Reloaded, not merely imported. A kernel that imported workshop_config before
# the file was edited keeps serving the cached module, and the first name added
# since then fails much further down as a bare NameError -- which is exactly how
# `raw_frames()` broke for anyone whose kernel predated it.
import workshop_config
importlib.reload(workshop_config)
from workshop_config import *   # noqa: F403  (RAW_DIR, WORK_DIR, PHASE*_DIR, ...)

require_dataset()   # fails now, with the command that fixes it, not later
os.makedirs(WORK_DIR, exist_ok=True)
show_config()

## What Phase 1 will consume
Before running calibration (next notebook), inventory the raw frames by type
(science / flat / dark / bias) and check their FITS structure.

In [ ]:
import os
from collections import Counter
from astropy.io import fits

# `raw_frames()` walks RAW_DIR, so the acquisition software's night tree
# (<date>/BIAS|DARK|FLAT|LIGHT/<target>/) and a flat simulated directory are
# both read the same way -- see workshop_config.raw_frames.
raw_files = raw_frames()
print(f"Found {len(raw_files)} raw files under {RAW_DIR!r}\n")

def classify(img_type):
    img_type = (img_type or '').lower()
    if 'bias' in img_type: return 'bias'
    if 'dark' in img_type: return 'dark'
    if 'flat' in img_type: return 'flat'
    return 'science'  # the profile decides this; see get_profile(INSTRUMENT).get_image_type

def folder_kind(path):
    """The type this frame's folder claims, or None in a flat directory."""
    for part in os.path.relpath(path, RAW_DIR).split(os.sep)[:-1]:
        name = part.lower()
        if name in ('bias', 'dark', 'flat'):
            return name
        if name == 'light':
            return 'science'
    return None

type_counts = Counter()
breakdown = Counter()
n_hdus = Counter()
ext_names = Counter()
by_folder = Counter()
misfiled = []
sci_files = []

for f in raw_files:
    with fits.open(f) as hdul:
        hdr = hdul[0].header
        kind = classify(hdr.get('IMAGETYP'))
        filt = hdr.get('FILTER', '<none>')
        exptime = hdr.get('EXPTIME', '<none>')
        type_counts[kind] += 1
        breakdown[(kind, str(filt), str(exptime))] += 1
        n_hdus[len(hdul)] += 1
        for hdu in hdul:
            ext_names[hdu.name] += 1
        by_folder[(os.path.relpath(os.path.dirname(f), RAW_DIR), kind)] += 1
        # The folder a frame sits in is a filing convention; IMAGETYP is the
        # fact, and the fact is what the pipeline reduces on. A disagreement is
        # harmless but worth seeing.
        folder_says = folder_kind(f)
        if folder_says is not None and folder_says != kind:
            misfiled.append((os.path.relpath(f, RAW_DIR), folder_says, kind))
        if kind == 'science':
            sci_files.append(f)

print('Frame counts by type:')
for k in ('science', 'flat', 'dark', 'bias'):
    print(f'  {k:>8}: {type_counts.get(k, 0)}')

print('\nWhere they live (directory -> type from the header):')
for (relative, kind), count in sorted(by_folder.items()):
    print(f'  {relative:<40s} {kind:>8}: {count}')
if misfiled:
    print('\nFolder and IMAGETYP disagree for:')
    for name, folder_says, header_says in misfiled[:10]:
        print(f'  {name}: folder says {folder_says}, header says {header_says}')

print('\nBreakdown by (type, filter, exptime):')
for k, v in sorted(breakdown.items()):
    print(f'  {k}: {v}')

print('\nHDU count per file (extensions present):')
for k, v in sorted(n_hdus.items()):
    print(f'  {k} HDU(s): {v} files')
print('Extension names seen:', dict(ext_names))

bpm_candidates = [f for f in raw_files if any(tag in os.path.basename(f).lower()
                                                for tag in ('badpix', 'bpm', 'mask'))]
print('\nStandalone bad-pixel-mask file present:', bool(bpm_candidates), bpm_candidates)
print('(cassa-photometry builds the bad-pixel mask on the fly from the master')
print(' flats during Phase 1 -- it is not a file you provide.)')

if sci_files:
    print(f"\nScience frame example: {os.path.relpath(sci_files[0], RAW_DIR)}")
    with fits.open(sci_files[0]) as hdul:
        hdul.info()

## Are these frames worth reducing?

Phase 1 will reduce whatever you give it. It does not refuse bad calibration
frames, and nothing downstream can tell afterwards: a saturated flat divides out
to a plausible-looking image whose photometry is quietly meaningless, and the
first symptom is a zero point that fails two notebooks later, for a reason that
no longer points back here.

So check the frames before spending an hour reducing them. The cells below
measure every frame once, then test each measurement against what Phase 1
actually needs from it. **FAIL** means fix it before reducing; **WARN** means
the pipeline will cope but the result is weaker than it looks; a check with
nothing to say says nothing.

In [ ]:
import os
import numpy as np
import pandas as pd
import textwrap
from astropy.io import fits
from astropy.stats import mad_std

from cassa_photometry.config import load_config
from cassa_photometry.instruments import get_profile

cfg = load_config()
instrument = get_profile(cfg.instrument, config=cfg)

#: The highest value the digitisation can produce -- what a frame really clips
#: against. Not the same as the full well a header quotes, which can be in
#: another unit or simply wrong; both are checked below.
ADC_CEILING = {8: 255.0, 16: 65535.0, 32: 2.0 ** 31 - 1}


def measure(path):
    """One pass over a frame: everything the checks need, and nothing more."""
    with fits.open(path) as hdul:
        header = hdul[0].header
        data = np.asarray(hdul[0].data, dtype=np.float64)

    ceiling = ADC_CEILING.get(int(header.get('BITPIX', 0)))   # None for float data
    finite = data[np.isfinite(data)]
    return {
        'file': os.path.relpath(path, RAW_DIR),
        'kind': instrument.get_image_type(header),
        'filter': instrument.get_filter(header),
        'exptime': instrument.get_exposure(header),
        'temp': header.get('CCD-TEMP', header.get('SET-TEMP')),
        'shape': f"{header.get('NAXIS1')}x{header.get('NAXIS2')}",
        'binning': f"{header.get('XBINNING')}x{header.get('YBINNING')}",
        'median': float(np.median(finite)) if finite.size else np.nan,
        # Robust scatter. A frame clipped flat against the ceiling has none, and
        # that absence is the single most useful number here.
        'spread': float(mad_std(finite)) if finite.size else np.nan,
        'sat_frac': float(np.mean(finite >= 0.999 * ceiling)) if ceiling else np.nan,
        'ceiling': ceiling,
        'precal': instrument.already_calibrated(header) or '',
        'gain': instrument.get_gain(header),
        'read_noise': instrument.get_read_noise(header),
        'saturate': instrument.get_saturation(header),
        'pointed': any(header.get(k) is not None for k in ('OBJCTRA', 'RA', 'CRVAL1')),
    }


frames = pd.DataFrame([measure(p) for p in raw_frames()])
print(f"Measured {len(frames)} frames.\n")

print(frames.groupby(['kind', 'filter']).agg(
    n=('file', 'size'),
    exptime=('exptime', 'median'),
    median_ADU=('median', 'median'),
    scatter_ADU=('spread', 'median'),
    pct_saturated=('sat_frac', lambda s: 100 * s.max()),
    temp_C=('temp', 'median'),
).to_string(float_format=lambda v: f'{v:,.2f}'))

In [ ]:
import matplotlib.pyplot as plt

# One picture of the whole night. Watch the ceiling line: any population sitting
# on it is clipped, and a clipped flat is not a flat.
#
# Each type gets its own column, because a bias and a dark differ by a couple of
# ADU and would otherwise hide each other completely.
fig, ax = plt.subplots(figsize=(9.5, 5))
colours = {'bias': '#4c72b0', 'dark': '#55a868', 'flat': '#dd8452', 'science': '#c44e52'}
order = [k for k in ('bias', 'dark', 'flat', 'science') if k in set(frames['kind'])]

ceiling = frames['ceiling'].dropna()
ceiling = float(ceiling.iloc[0]) if len(ceiling) else None
if ceiling:
    ax.axhspan(0.33 * ceiling, 0.67 * ceiling, color='seagreen', alpha=0.12, zorder=0)
    ax.axhline(ceiling, color='crimson', ls='--', lw=1.2, zorder=1,
               label=f'ADC ceiling ({ceiling:,.0f} ADU)')
    ax.text(len(order) - 0.45, 0.47 * ceiling, 'healthy flat level',
            color='seagreen', fontsize=9, ha='right', va='center')

rng = np.random.default_rng(0)
for x, kind in enumerate(order):
    group = frames[frames['kind'] == kind]
    jitter = rng.uniform(-0.22, 0.22, len(group))    # spread ties apart
    ax.scatter(x + jitter, group['median'], s=30, alpha=0.8, zorder=3,
               color=colours.get(kind, 'grey'), edgecolor='white', linewidth=0.4)

# Counts go in the tick labels: nothing can collide with them there.
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{k}\nn={int((frames['kind'] == k).sum())}" for k in order])
ax.set_yscale('log')
ax.set_ylabel('median level (ADU)')
ax.set_title('Signal level of every raw frame')
ax.legend(frameon=False, loc='center left')
plt.tight_layout()
plt.show()

In [ ]:
from cassa_photometry.phase1_calibration.pipeline import MIN_FRAMES_FOR_EMPIRICAL_SIGMA

findings = []


def check(level, headline, detail=''):
    """Record one finding. FAIL blocks the reduction, WARN weakens it."""
    findings.append((level, headline, detail))


by_kind = {k: g for k, g in frames.groupby('kind')}
science, bias = by_kind.get('science'), by_kind.get('bias')
dark, flat = by_kind.get('dark'), by_kind.get('flat')
ceiling = frames['ceiling'].dropna()
ceiling = float(ceiling.iloc[0]) if len(ceiling) else None

# --- Is there anything to reduce, and enough to reduce it with? --------------
if science is None:
    check('FAIL', 'No science frames',
          "Nothing carries a science IMAGETYP. Phase 1 has nothing to calibrate.")
else:
    for name, group in (('bias', bias), ('dark', dark), ('flat', flat)):
        if group is None:
            check('WARN', f'No {name} frames',
                  f'Phase 1 will skip {name} correction entirely, and every error bar '
                  f'downstream is then optimistic because it omits that term.')
        elif name != 'flat' and len(group) < MIN_FRAMES_FOR_EMPIRICAL_SIGMA:
            check('WARN', f'Only {len(group)} {name} frame(s)',
                  f'Below {MIN_FRAMES_FOR_EMPIRICAL_SIGMA} frames the frame-to-frame '
                  f'scatter cannot be measured, so the master carries an estimated '
                  f'uncertainty rather than an observed one.')

    # --- A flat for every filter that was actually observed -----------------
    science_filters = set(science['filter'])
    flat_filters = set(flat['filter']) if flat is not None else set()
    proxies = {k.upper(): v.upper() for k, v in instrument.flat_proxies().items()}
    for filt in sorted(science_filters - flat_filters):
        proxy = proxies.get(filt.upper())
        if proxy and proxy in {f.upper() for f in flat_filters}:
            check('WARN', f'No {filt} flat; the profile substitutes {proxy}',
                  'Sound when the two filters share a response, wrong when they do not.')
        else:
            check('FAIL', f'No flat for science filter {filt}',
                  'Those frames go unflattened, so the zero point varies across the '
                  'field and the photometry is not comparable from corner to corner.')
    for filt in sorted(flat_filters & science_filters):
        n = int((flat['filter'] == filt).sum())
        if n < MIN_FRAMES_FOR_EMPIRICAL_SIGMA:
            check('WARN', f'Only {n} flat(s) in {filt}',
                  'The master flat then carries one frame\'s noise into every science '
                  'frame it divides.')

    # --- The flats: the check that catches the most damaging mistake --------
    if flat is not None and ceiling:
        worst = flat.loc[flat['sat_frac'].idxmax()]
        if worst['sat_frac'] > 0.01:
            check('FAIL', f"Flats are saturated ({100 * worst['sat_frac']:.0f}% of pixels at the ADC ceiling)",
                  f'A clipped flat records no response at all: every pixel reads the '
                  f'same number, so dividing by it corrects nothing and multiplies '
                  f'noise instead. Re-take them shorter -- aim for a median near '
                  f'{0.4 * ceiling:,.0f} ADU against the {ceiling:,.0f} ADU ceiling.')
        elif worst['median'] > 0.9 * ceiling:
            check('WARN', f"Flats sit at {worst['median']:,.0f} ADU, near the {ceiling:,.0f} ADU ceiling",
                  'The brightest pixels are probably clipping already. Shorten the exposure.')

        # A flat with no structure carries nothing even when it is not clipping:
        # no vignetting, no dust shadows, not even photon noise.
        relative = (flat['spread'] / flat['median'].replace(0, np.nan)).median()
        if float(relative) < 0.002:
            check('FAIL', 'Flats have almost no pixel-to-pixel structure',
                  f'Relative scatter is {float(relative):.1e}: no vignetting, no dust, '
                  f'not even photon noise. This cannot describe a detector response.')

        bias_level = float(bias['median'].median()) if bias is not None else 0.0
        signal = float(flat['median'].median()) - bias_level
        if 0 < signal < 0.1 * ceiling:
            check('WARN', f'Flats are faint ({signal:,.0f} ADU above bias)',
                  'A dim flat is mostly its own photon noise, which it then injects '
                  'into every science frame.')

    # --- Bias ---------------------------------------------------------------
    if bias is not None:
        level = float(bias['median'].median())
        if level <= 0:
            check('FAIL', f'Bias median is {level:,.0f} ADU',
                  'A pedestal at or below zero is clipped, and everything measured '
                  'against it is biased low.')
        elif ceiling and level > 0.2 * ceiling:
            check('WARN', f'Bias pedestal is high ({level:,.0f} ADU)',
                  'It spends dynamic range for no benefit.')

        gains, noises = bias['gain'].dropna(), bias['read_noise'].dropna()
        if len(gains) and len(noises) and float(gains.iloc[0]):
            expected = float(noises.iloc[0]) / float(gains.iloc[0])
            measured = float(bias['spread'].median())
            if expected > 0 and not 0.3 <= measured / expected <= 3.0:
                check('WARN', f'Bias scatter is {measured:.2f} ADU; the header read noise implies {expected:.2f}',
                      'One of the two is wrong, and the header value is what Phase 1 '
                      'writes into every ERR plane.')

    # --- Dark ---------------------------------------------------------------
    if dark is not None and bias is not None:
        excess = float(dark['median'].median()) - float(bias['median'].median())
        if excess < -float(bias['spread'].median()):
            check('WARN', f'Darks read {excess:.1f} ADU BELOW the bias',
                  'Subtracting a scaled negative dark adds signal instead of removing it.')
        dark_exp, sci_exp = float(dark['exptime'].median()), float(science['exptime'].median())
        if dark_exp and sci_exp and dark_exp < 0.5 * sci_exp:
            check('WARN', f'Darks are {dark_exp:g}s against {sci_exp:g}s science frames',
                  'Phase 1 scales the dark by the exposure ratio, and scales its noise '
                  'up with it. Match the exposures where you can.')

    # --- Conditions ---------------------------------------------------------
    temps = frames['temp'].dropna().astype(float)
    if len(temps) and (temps.max() - temps.min()) > 2.0:
        check('WARN', f'Sensor temperature spans {temps.min():.1f} to {temps.max():.1f} C',
              'Dark current roughly doubles every 6 C, so a dark taken warm does not '
              'describe a science frame taken cold.')

    # --- Science ------------------------------------------------------------
    if bias is not None:
        sky = float(science['median'].median()) - float(bias['median'].median())
        if sky <= 0:
            check('FAIL', 'Science frames hold no signal above the bias pedestal',
                  'The shutter, the exposure or the pointing failed.')
        elif sky < 5 * float(bias['spread'].median()):
            check('WARN', f'Sky is only {sky:,.0f} ADU above bias',
                  'These frames are read-noise limited; longer exposures buy real depth.')

    hot = float(science['sat_frac'].max())
    if hot > 0.05:
        check('WARN', f'{100 * hot:.1f}% of pixels saturate in the worst science frame',
              'Saturated cores cannot be photometered and they distort the PSF fit.')

    if not science['pointed'].all():
        check('WARN', f"{int((~science['pointed']).sum())} science frame(s) carry no RA/Dec",
              'Phase 2 must then solve blind, which is far slower and often fails.')

# --- Do the headers describe the frames they are attached to? ---------------
saturate = frames['saturate'].dropna()
if ceiling and len(saturate) and float(saturate.max()) > ceiling:
    check('WARN', f'SATURATE={float(saturate.max()):,.0f} exceeds the {ceiling:,.0f} ADU ceiling',
          'No pixel can reach it, so the DQ saturation flag never fires and clipped '
          'stars pass as good measurements.')

if frames['gain'].isna().any() or frames['read_noise'].isna().any():
    check('WARN', 'Some frames state no gain or read noise',
          f'Phase 1 falls back to the configured {cfg.phase1.fallback_gain} e-/ADU, '
          f'a guess that every error bar then inherits.')

precal = frames[frames['precal'] != '']
if len(precal):
    check('FAIL', f'{len(precal)} frame(s) report prior calibration',
          f"e.g. {precal.iloc[0]['file']} ({precal.iloc[0]['precal']}). Phase 1 skips "
          f"these by default: reducing a reduced frame yields a believable image with "
          f"a wrong error budget.")

if frames['shape'].nunique() > 1 or frames['binning'].nunique() > 1:
    check('WARN', 'Frames have mixed geometry or binning',
          f"shapes {sorted(frames['shape'].unique())}, "
          f"binning {sorted(frames['binning'].unique())}. A master only applies to "
          f"frames of its own geometry.")

# --- Verdict -----------------------------------------------------------------
fails = [f for f in findings if f[0] == 'FAIL']
warns = [f for f in findings if f[0] == 'WARN']

print('=' * 78)
for level, headline, detail in fails + warns:
    print(f'[{level}] {headline}')
    for line in textwrap.wrap(detail, 72):
        print(f'        {line}')
    print()

print('=' * 78)
if fails:
    print(f'STOP: {len(fails)} blocking problem(s), {len(warns)} warning(s).')
    print('Reducing these frames produces numbers that look right and are not.')
    print('Fix the FAILs above, then re-run this notebook before going on to 02.')
elif warns:
    print(f'PROCEED WITH CAUTION: {len(warns)} warning(s), nothing blocking.')
    print('Phase 1 will cope, but the result is weaker than it looks -- keep the')
    print('warnings above in mind when reading the final error bars.')
else:
    print('PROCEED: every check passed. On to notebook 02.')
print('=' * 78)

### Reading the verdict

A **FAIL** is not a suggestion. Each one names a specific way the reduction will
be *wrong* rather than merely noisy, and none of them can be repaired later from
the reduced products -- the information was never recorded in the first place.

| Finding | What to do about it |
|---|---|
| Flats saturated, or with no structure | Re-take them. Aim for a median between a third and two thirds of the ADC ceiling: bright enough that photon noise is small, far enough from clipping that no pixel reaches it. |
| No flat for a science filter | Take one in that filter, or accept that those frames go unflattened and their photometry varies across the field. |
| Fewer than three calibration frames | Take more. Below three the master's uncertainty is estimated rather than measured, and every `ERR` plane inherits that. |
| Prior calibration reported | Go back to the original raw frames. Reducing a reduced frame gives a believable image with a wrong error budget. |
| Temperature drift | Match the darks to the science frames' temperature; dark current roughly doubles every 6 C. |
| `SATURATE` above the ADC ceiling | Fix the header, or set `phase1.saturation_adu` in the config. Until then the `DQ` saturation flag can never fire and clipped stars pass as good measurements. |

If the verdict says stop, fixing the frames is the cheap path. The alternative
is finding out in notebook 04, when the zero point fails for a reason that is
four steps upstream.